# 03 — Prepare ML Dataset (V9 forecast traffic-memory)

**Phase 2.1 — TFE Phuket** | Avril 2026

**Branch:** `v9_forecast_traffic_memory_scratch`

**Why this V9 exists:** les versions précédentes textualisaient surtout des signaux exogènes et pas assez de mémoire trafic. Cette V9 ajoute explicitement des **lags trafic courts (l1/l2/l3)** pour que le pipeline forecast voie enfin les variables qui ressortaient comme les plus prédictives dans les baselines ML et SHAP.

**Input:** `data/phuket_master.csv` (96 rows × 101 cols)

**Outputs:**
- `data/ml_dataset.csv` — all 96 rows + lag features + split label
- `data/ml_train.csv` — training split (2023, NaN lags dropped)
- `data/ml_val.csv` — validation split (Jan–Jun 2024)
- `data/ml_test.csv` — test split (Jul–Dec 2024)

**V9 design choices:**
1. Lags `l1/l2/l3` for exogenous anchors **and** forecast-critical traffic memory
2. Traffic-memory vars added for forecast: `tt_ratio_Weekday_AM1`, `pti_Weekday_AM1`, `spd_Weekday_AM1`, `tt_ratio_Weekday_AM2`, `tt_ratio_Weekday_PrePM`, `tt_ratio_Weekend_AM1`
3. Temporal split stays strictly forward-in-time to avoid leakage
4. `ml_dataset.csv` remains the canonical bridge toward `04_textualize.ipynb` and `06_llm_finetuning.ipynb`
5. Traffic values remain fully dynamic (TomTom monthly values, 2023-2024)


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded.')
print(f'Pandas {pd.__version__} | NumPy {np.__version__}')

Libraries loaded.
Pandas 3.0.0 | NumPy 2.4.2


In [2]:
# ── Load master table ──────────────────────────────────────────────────────────
df = pd.read_csv('../data/phuket_master.csv')

print('=' * 70)
print('MASTER TABLE LOADED')
print('=' * 70)
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Missing values: {df.isna().sum().sum()}')
print(f'Years: {sorted(df["year"].unique())}')
print(f'Corridors ({df["corr_id"].nunique()}):')
for cid in sorted(df['corr_id'].unique()):
    name = df[df['corr_id'] == cid]['corridor'].iloc[0]
    print(f'  [{cid}] {name}')

print()
print('Target variable descriptive stats:')
targets_preview = ['tt_ratio_Weekday_AM1', 'tt_ratio_Weekday_PM1', 'pti_mean', 'pti_max']
print(df[targets_preview].describe().round(3))
print(df[targets_preview].describe().round(4).to_string())

print()
print('First 4 rows (identifiers + targets):')
display(df[['year', 'month', 'corr_id', 'corridor', 'season',
            'tt_ratio_Weekday_AM1', 'tt_ratio_Weekday_PM1', 'tt_ratio_Weekday_Midday', 'pti_mean']].head(4))

MASTER TABLE LOADED
Shape: 96 rows × 101 columns
Missing values: 0
Years: [np.int64(2023), np.int64(2024)]
Corridors (4):
  [0] Airport Road (Route 402)
  [1] Patong Hill (Route 4029)
  [2] Phuket Town → Rawai (Route 4022)
  [3] Bypass Road (Route 4027)

Target variable descriptive stats:
       tt_ratio_Weekday_AM1  tt_ratio_Weekday_PM1  pti_mean  pti_max
count               96.0000               96.0000   96.0000  96.0000
mean                 1.4000                1.5740    2.8030   3.8770
std                  0.2270                0.2130    0.5130   0.8090
min                  0.9900                1.1900    2.1540   2.7300
25%                  1.2300                1.4380    2.4600   3.2780
50%                  1.3150                1.5500    2.5580   3.6650
75%                  1.6120                1.6750    3.1070   4.2520
max                  1.8400                2.0700    3.9990   5.6800
       tt_ratio_Weekday_AM1  tt_ratio_Weekday_PM1  pti_mean  pti_max
count               

,year,month,corr_id,corridor,season,tt_ratio_Weekday_AM1,tt_ratio_Weekday_PM1,tt_ratio_Weekday_Midday,pti_mean
0,2023,1,0,Airport Road (Route 402),high_peak,1.1400,1.4800,1.3600,2.3890
1,2023,1,1,Patong Hill (Route 4029),high_peak,1.1300,1.2900,1.1800,2.4510
2,2023,1,2,Phuket Town → Rawai (Route 4022),high_peak,1.5400,1.9300,1.5600,3.3750
3,2023,1,3,Bypass Road (Route 4027),high_peak,1.4600,1.4900,1.3300,2.3180


In [3]:
# ── Sort + verify temporal coverage ───────────────────────────────────────────
df = df.sort_values(['corr_id', 'year', 'month']).reset_index(drop=True)

print('=' * 70)
print('TEMPORAL COVERAGE PER CORRIDOR (after sort by [corr_id, year, month])')
print('=' * 70)

for cid in sorted(df['corr_id'].unique()):
    sub = df[df['corr_id'] == cid]
    name = sub['corridor'].iloc[0]
    seasons = dict(sub['season'].value_counts())
    monsoon_months = sub[sub['is_monsoon'] == 1]['month'].tolist()
    print(f'\nCorridor {cid} — {name}')
    print(f'  Rows: {len(sub)}')
    print(f'  Range: {sub["year"].min()}-{sub["month"].min():02d} → {sub["year"].max()}-{sub["month"].max():02d}')
    print(f'  Seasons: {seasons}')
    print(f'  Monsoon months (is_monsoon=1): {sorted(set(monsoon_months))}')

# Verify no gaps
print()
print('Temporal continuity check (per corridor):')
for cid in sorted(df['corr_id'].unique()):
    sub = df[df['corr_id'] == cid][['year', 'month']].copy()
    sub['date'] = pd.to_datetime(sub[['year', 'month']].assign(day=1))
    sub = sub.sort_values('date')
    expected = set(pd.date_range(sub['date'].min(), sub['date'].max(), freq='MS').strftime('%Y-%m'))
    actual = set(sub['date'].dt.strftime('%Y-%m'))
    gaps = expected - actual
    print(f'  [{cid}] Gaps: {gaps if gaps else "NONE ✓"}')

TEMPORAL COVERAGE PER CORRIDOR (after sort by [corr_id, year, month])

Corridor 0 — Airport Road (Route 402)
  Rows: 24
  Range: 2023-01 → 2024-12
  Seasons: {'monsoon': np.int64(10), 'high_peak': np.int64(8), 'shoulder': np.int64(6)}
  Monsoon months (is_monsoon=1): [5, 6, 7, 8, 9, 10]

Corridor 1 — Patong Hill (Route 4029)
  Rows: 24
  Range: 2023-01 → 2024-12
  Seasons: {'monsoon': np.int64(10), 'high_peak': np.int64(8), 'shoulder': np.int64(6)}
  Monsoon months (is_monsoon=1): [5, 6, 7, 8, 9, 10]

Corridor 2 — Phuket Town → Rawai (Route 4022)
  Rows: 24
  Range: 2023-01 → 2024-12
  Seasons: {'monsoon': np.int64(10), 'high_peak': np.int64(8), 'shoulder': np.int64(6)}
  Monsoon months (is_monsoon=1): [5, 6, 7, 8, 9, 10]

Corridor 3 — Bypass Road (Route 4027)
  Rows: 24
  Range: 2023-01 → 2024-12
  Seasons: {'monsoon': np.int64(10), 'high_peak': np.int64(8), 'shoulder': np.int64(6)}
  Monsoon months (is_monsoon=1): [5, 6, 7, 8, 9, 10]

Temporal continuity check (per corridor):
  [0] G

In [4]:
# ── Column group definitions ───────────────────────────────────────────────────
IDENTIFIERS = ['year', 'month', 'corridor', 'corr_id', 'season',
               'is_high_season', 'is_monsoon', 'month_sin', 'month_cos']

# Target variables (main target: tt_ratio_Weekday_AM1 — 4 representative timesets)
# Congestion_* columns removed — use tt_ratio + pti as targets
TARGETS = ['tt_ratio_Weekday_AM1', 'tt_ratio_Weekday_PM1',
           'tt_ratio_Weekday_Midday', 'tt_ratio_Weekend_AM1',
           'pti_mean', 'pti_max']
PRIMARY = 'tt_ratio_Weekday_AM1'

# TomTom traffic columns — now DYNAMIC (real monthly values, not static profile)
TRAFFIC_FEATURES = (
    [c for c in df.columns if c.startswith('spd_')]          +
    [c for c in df.columns if c.startswith('tt_ratio')]      +
    [c for c in df.columns if c.startswith('pti_')]          +
    ['corr_tourist_access', 'corr_length_km']
)
_seen = set()
TRAFFIC_FEATURES = [c for c in TRAFFIC_FEATURES if not (_seen.add(c) or c in _seen - {c})]

# Exogenous temporal features
FLIGHTS_ALL  = ['flt_intl_arrivals', 'flt_intl_departures', 'flt_total_pax', 'flt_domestic']
WEATHER_ALL  = ['wx_temp_c_mean', 'wx_rain_mm_sum', 'wx_rain_days', 'wx_wind_kmh']
SOCIAL_ALL   = ['soc_phuket', 'soc_phuket_beach', 'soc_phuket_flight', 'soc_phuket_hotel', 'soc_phuket_vacation']
CALENDAR_ALL = ['cal_n_holidays', 'cal_n_events', 'cal_high_impact']
EXO_ALL      = FLIGHTS_ALL + WEATHER_ALL + SOCIAL_ALL + CALENDAR_ALL

print('=' * 70)
print('COLUMN GROUPS')
print('=' * 70)
print(f'Identifiers      ({len(IDENTIFIERS):2d}): {IDENTIFIERS}')
print(f'Targets          ({len(TARGETS):2d}): {TARGETS}')
print(f'Traffic features ({len(TRAFFIC_FEATURES):2d}): {TRAFFIC_FEATURES[:6]}...')
print(f'Flights ALL      ({len(FLIGHTS_ALL):2d}): {FLIGHTS_ALL}')
print(f'Weather ALL      ({len(WEATHER_ALL):2d}): {WEATHER_ALL}')
print(f'Social ALL       ({len(SOCIAL_ALL):2d}): {SOCIAL_ALL}')
print(f'Calendar ALL     ({len(CALENDAR_ALL):2d}): {CALENDAR_ALL}')


COLUMN GROUPS
Identifiers      ( 9): ['year', 'month', 'corridor', 'corr_id', 'season', 'is_high_season', 'is_monsoon', 'month_sin', 'month_cos']
Targets          ( 6): ['tt_ratio_Weekday_AM1', 'tt_ratio_Weekday_PM1', 'tt_ratio_Weekday_Midday', 'tt_ratio_Weekend_AM1', 'pti_mean', 'pti_max']
Traffic features (76): ['spd_Weekday_AM1', 'spd_Weekday_AM2', 'spd_Weekday_Afternoon', 'spd_Weekday_Early', 'spd_Weekday_Evening', 'spd_Weekday_LateNight']...
Flights ALL      ( 4): ['flt_intl_arrivals', 'flt_intl_departures', 'flt_total_pax', 'flt_domestic']
Weather ALL      ( 4): ['wx_temp_c_mean', 'wx_rain_mm_sum', 'wx_rain_days', 'wx_wind_kmh']
Social ALL       ( 5): ['soc_phuket', 'soc_phuket_beach', 'soc_phuket_flight', 'soc_phuket_hotel', 'soc_phuket_vacation']
Calendar ALL     ( 3): ['cal_n_holidays', 'cal_n_events', 'cal_high_impact']


In [5]:
# ── Lag features L1 / L2 / L3 (computed per corridor) ─────────────────────────
# V9 review insight:
# forecast quality appears to depend much more on traffic-internal memory than on
# long exogenous narration alone. We therefore keep the original exogenous lags
# but add forecast-critical traffic lags as well.
EXO_LAG_VARS = ['flt_total_pax', 'wx_rain_mm_sum', 'soc_phuket', 'cal_n_holidays']
TRAFFIC_LAG_VARS = [
    'tt_ratio_Weekday_AM1',
    'pti_Weekday_AM1',
    'spd_Weekday_AM1',
    'tt_ratio_Weekday_AM2',
    'tt_ratio_Weekday_PrePM',
    'tt_ratio_Weekend_AM1',
]
LAG_VARS   = EXO_LAG_VARS + TRAFFIC_LAG_VARS
LAG_DEPTHS = [1, 2, 3]

lag_cols = []
for var in LAG_VARS:
    for lag in LAG_DEPTHS:
        col = f'{var}_l{lag}'
        df[col] = df.groupby('corr_id')[var].shift(lag)
        lag_cols.append(col)

print('=' * 70)
print('LAG FEATURES COMPUTED')
print('=' * 70)
print(f'Exogenous lag vars : {EXO_LAG_VARS}')
print(f'Traffic lag vars   : {TRAFFIC_LAG_VARS}')
print(f'Lag depths         : {LAG_DEPTHS}')
print(f'New columns        : {len(lag_cols)}')
print(f'New shape          : {df.shape[0]} rows × {df.shape[1]} cols')

print()
print('NaN counts per lag column:')
for col in lag_cols:
    n = df[col].isna().sum()
    print(f'  {col:<30s}: {n} NaN  ({n/len(df)*100:.1f}%)')

print()
print('Sample: first 5 rows of corridor 0 showing lag pattern:')
c0 = df[df['corr_id'] == 0].head(5)[['year', 'month'] + lag_cols[:9]]
display(c0)


LAG FEATURES COMPUTED
Exogenous lag vars : ['flt_total_pax', 'wx_rain_mm_sum', 'soc_phuket', 'cal_n_holidays']
Traffic lag vars   : ['tt_ratio_Weekday_AM1', 'pti_Weekday_AM1', 'spd_Weekday_AM1', 'tt_ratio_Weekday_AM2', 'tt_ratio_Weekday_PrePM', 'tt_ratio_Weekend_AM1']
Lag depths         : [1, 2, 3]
New columns        : 30
New shape          : 96 rows × 131 cols

NaN counts per lag column:
  flt_total_pax_l1              : 4 NaN  (4.2%)
  flt_total_pax_l2              : 8 NaN  (8.3%)
  flt_total_pax_l3              : 12 NaN  (12.5%)
  wx_rain_mm_sum_l1             : 4 NaN  (4.2%)
  wx_rain_mm_sum_l2             : 8 NaN  (8.3%)
  wx_rain_mm_sum_l3             : 12 NaN  (12.5%)
  soc_phuket_l1                 : 4 NaN  (4.2%)
  soc_phuket_l2                 : 8 NaN  (8.3%)
  soc_phuket_l3                 : 12 NaN  (12.5%)
  cal_n_holidays_l1             : 4 NaN  (4.2%)
  cal_n_holidays_l2             : 8 NaN  (8.3%)
  cal_n_holidays_l3             : 12 NaN  (12.5%)
  tt_ratio_Weekday_AM1_l

,year,month,flt_total_pax_l1,flt_total_pax_l2,flt_total_pax_l3,wx_rain_mm_sum_l1,wx_rain_mm_sum_l2,wx_rain_mm_sum_l3,soc_phuket_l1,soc_phuket_l2,soc_phuket_l3
0,2023,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,2,1258673.0000,NaN,NaN,295.0000,NaN,NaN,70.0000,NaN,NaN
2,2023,3,1211138.0000,1258673.0000,NaN,128.6000,295.0000,NaN,64.5000,70.0000,NaN
3,2023,4,1332891.0000,1211138.0000,1258673.0000,205.8000,128.6000,295.0000,60.0000,64.5000,70.0000
4,2023,5,1185581.0000,1332891.0000,1211138.0000,251.4000,205.8000,128.6000,55.0000,60.0000,64.5000


In [6]:
# ── NaN pattern analysis ───────────────────────────────────────────────────────
nan_mask = df[lag_cols].isna().any(axis=1)

print('=' * 70)
print('NaN PATTERN — FIRST 3 MONTHS PER CORRIDOR')
print('=' * 70)
print(f'Rows with ANY NaN lag: {nan_mask.sum()} / {len(df)}')
print()

nan_rows = df[nan_mask][['year', 'month', 'corr_id', 'corridor'] + lag_cols]
print('Affected rows (one per month/corridor):')
display(nan_rows[['year', 'month', 'corr_id', 'flt_total_pax_l1', 'flt_total_pax_l2', 'flt_total_pax_l3']])

print()
print('Impact on datasets:')
print(f'  LLM dataset (llm_prompts.jsonl) : KEEPS all {len(df)} rows')
print(f'    → HISTORY block left partially empty for Jan/Feb/Mar 2023')
print(f'  ML dataset  (ml_train/val/test)  : DROPS {nan_mask.sum()} rows')
print(f'    → {len(df) - nan_mask.sum()} ML-ready rows remain')
print(f'    → First usable month: April 2023 (all 3 lags available)')

NaN PATTERN — FIRST 3 MONTHS PER CORRIDOR
Rows with ANY NaN lag: 12 / 96

Affected rows (one per month/corridor):


,year,month,corr_id,flt_total_pax_l1,flt_total_pax_l2,flt_total_pax_l3
0,2023,1,0,NaN,NaN,NaN
1,2023,2,0,1258673.0000,NaN,NaN
2,2023,3,0,1211138.0000,1258673.0000,NaN
24,2023,1,1,NaN,NaN,NaN
25,2023,2,1,1258673.0000,NaN,NaN
26,2023,3,1,1211138.0000,1258673.0000,NaN
48,2023,1,2,NaN,NaN,NaN
49,2023,2,2,1258673.0000,NaN,NaN
50,2023,3,2,1211138.0000,1258673.0000,NaN
72,2023,1,3,NaN,NaN,NaN



Impact on datasets:
  LLM dataset (llm_prompts.jsonl) : KEEPS all 96 rows
    → HISTORY block left partially empty for Jan/Feb/Mar 2023
  ML dataset  (ml_train/val/test)  : DROPS 12 rows
    → 84 ML-ready rows remain
    → First usable month: April 2023 (all 3 lags available)


In [7]:
# ── Temporal split assignment ──────────────────────────────────────────────────
# Data: 2023-01 → 2024-12 (24 months × 4 corridors = 96 rows)
# Train : 2023 Jan–Dec  (12 months × 4 = 48 rows — before lag drop: 36)
# Val   : 2024 Jan–Jun  ( 6 months × 4 = 24 rows)
# Test  : 2024 Jul–Dec  ( 6 months × 4 = 24 rows)

def assign_split(year, month):
    if year == 2023:
        return 'train'
    elif year == 2024 and month <= 6:
        return 'val'
    else:
        return 'test'

df['split'] = df.apply(lambda r: assign_split(r['year'], r['month']), axis=1)

print('=' * 70)
print('TEMPORAL SPLIT ASSIGNMENT')
print('=' * 70)
print('Boundaries:')
print('  Train : Jan 2023 – Dec 2023  (12 months × 4 corridors)')
print('  Val   : Jan 2024 – Jun 2024  ( 6 months × 4 corridors)')
print('  Test  : Jul 2024 – Dec 2024  ( 6 months × 4 corridors)')
print()
for s in ['train', 'val', 'test']:
    sub = df[df['split'] == s]
    print(f'  {s:5s}: {len(sub):3d} rows — '
          f'{sub.year.min()}-{sub.month.min():02d} → {sub.year.max()}-{sub.month.max():02d}')


TEMPORAL SPLIT ASSIGNMENT
Boundaries:
  Train : Jan 2023 – Dec 2023  (12 months × 4 corridors)
  Val   : Jan 2024 – Jun 2024  ( 6 months × 4 corridors)
  Test  : Jul 2024 – Dec 2024  ( 6 months × 4 corridors)

  train:  48 rows — 2023-01 → 2023-12
  val  :  24 rows — 2024-01 → 2024-06
  test :  24 rows — 2024-07 → 2024-12


In [8]:
# ── ML vs LLM feature sets ────────────────────────────────────────────────────
# ML: reduced for multicollinearity (XGBoost)
FLIGHTS_ML  = ['flt_total_pax']
WEATHER_ML  = ['wx_temp_c_mean', 'wx_rain_mm_sum', 'wx_wind_kmh']
SOCIAL_ML   = ['soc_phuket']
CALENDAR_ML = ['cal_n_holidays', 'cal_n_events', 'cal_high_impact']
EXO_ML      = FLIGHTS_ML + WEATHER_ML + SOCIAL_ML + CALENDAR_ML

# ML cyclical identifiers (drop corr string, season string — redundant)
ID_ML = ['month_sin', 'month_cos', 'is_high_season', 'is_monsoon', 'corr_id']

# ML / LLM lag features
LAG_COLS_ML = [f'{v}_l{l}' for v in LAG_VARS for l in LAG_DEPTHS]

ML_FEATURES = TRAFFIC_FEATURES + ID_ML + EXO_ML + LAG_COLS_ML

# Dropped from ML (kept in LLM prompt context)
DROPPED_ML = [c for c in EXO_ALL if c not in EXO_ML]

print('=' * 70)
print('FEATURE SETS: ML vs LLM')
print('=' * 70)

print(f'\n--- ML Feature Set (XGBoost/LightGBM) ---')
print(f'  Traffic features: {len(TRAFFIC_FEATURES)} cols (dynamic 2023-2024)')
print(f'  Identifiers    : {len(ID_ML)} cols → {ID_ML}')
print(f'  Exogenous      : {len(EXO_ML)} cols → {EXO_ML}')
print(f'  Lags           : {len(LAG_COLS_ML)} cols (exo + forecast-critical traffic memory)')
print(f'  TOTAL          : {len(ML_FEATURES)} features')

print(f'\n  Dropped from ML (multicollinearity reduction):')
reasons = {
    'flt_intl_arrivals'    : 'same monthly AOT report as flt_total_pax',
    'flt_intl_departures'  : 'same monthly AOT report as flt_total_pax',
    'flt_domestic'         : 'correlated with flt_total_pax',
    'wx_rain_days'         : 'r=0.979 with wx_rain_mm_sum (from EDA)',
    'soc_phuket_beach'     : 'sub-keyword of soc_phuket',
    'soc_phuket_flight'    : 'r=0.86 with flt_total_pax + soc_phuket',
    'soc_phuket_hotel'     : 'sub-keyword of soc_phuket',
    'soc_phuket_vacation'  : 'sub-keyword of soc_phuket',
}
for col in DROPPED_ML:
    r_val = df[col].corr(df['flt_total_pax'] if 'flt' in col else df['soc_phuket'] if 'soc' in col else df['wx_rain_mm_sum'])
    reason = reasons.get(col, 'multicollinear')
    print(f'  ✗ {col:<28s}: {reason}  (r={r_val:.3f})')

print(f'\n--- LLM Feature Set (Prompts) ---')
llm_features = list(dict.fromkeys(TRAFFIC_FEATURES + IDENTIFIERS + EXO_ALL + LAG_COLS_ML))
print(f'  ALL features kept: {len(llm_features)} cols')
print('  Rationale: V9 forecast prompting needs both current traffic state and traffic memory, not only exogenous context.')


FEATURE SETS: ML vs LLM

--- ML Feature Set (XGBoost/LightGBM) ---
  Traffic features: 76 cols (dynamic 2023-2024)
  Identifiers    : 5 cols → ['month_sin', 'month_cos', 'is_high_season', 'is_monsoon', 'corr_id']
  Exogenous      : 8 cols → ['flt_total_pax', 'wx_temp_c_mean', 'wx_rain_mm_sum', 'wx_wind_kmh', 'soc_phuket', 'cal_n_holidays', 'cal_n_events', 'cal_high_impact']
  Lags           : 30 cols (exo + forecast-critical traffic memory)
  TOTAL          : 119 features

  Dropped from ML (multicollinearity reduction):
  ✗ flt_intl_arrivals           : same monthly AOT report as flt_total_pax  (r=0.467)
  ✗ flt_intl_departures         : same monthly AOT report as flt_total_pax  (r=0.467)
  ✗ flt_domestic                : correlated with flt_total_pax  (r=0.537)
  ✗ wx_rain_days                : r=0.979 with wx_rain_mm_sum (from EDA)  (r=0.980)
  ✗ soc_phuket_beach            : sub-keyword of soc_phuket  (r=0.962)
  ✗ soc_phuket_flight           : r=0.86 with flt_total_pax + soc_phuke

In [9]:
# ── Correlation check (validates ML feature reduction) ────────────────────────
import matplotlib.pyplot as plt

print('=' * 70)
print('CORRELATION CHECK — JUSTIFICATION FOR ML FEATURE REDUCTION')
print('=' * 70)

# Check all pairwise correlations among EXO_ALL
corr_matrix = df[EXO_ALL].corr().abs()
high_corr = []
for i, c1 in enumerate(EXO_ALL):
    for j, c2 in enumerate(EXO_ALL):
        if i < j:
            r = corr_matrix.loc[c1, c2]
            if r > 0.5:
                high_corr.append({'var1': c1, 'var2': c2, 'r': r, 'dropped_var': c2 if c2 in DROPPED_ML else (c1 if c1 in DROPPED_ML else None)})

high_corr_df = pd.DataFrame(high_corr).sort_values('r', ascending=False)
print(f'Variable pairs with |r| > 0.5:')
print(high_corr_df[['var1', 'var2', 'r', 'dropped_var']].to_string(index=False))

print()
print('Target correlation (tt_ratio_Weekday_AM1):')
target_corr = df[EXO_ALL].corrwith(df['tt_ratio_Weekday_AM1']).abs().sort_values(ascending=False)
print(target_corr.round(4).to_string())

CORRELATION CHECK — JUSTIFICATION FOR ML FEATURE REDUCTION
Variable pairs with |r| > 0.5:
               var1                var2      r         dropped_var
  flt_intl_arrivals flt_intl_departures 0.9999 flt_intl_departures
     wx_rain_mm_sum        wx_rain_days 0.9801        wx_rain_days
  flt_intl_arrivals        flt_domestic 0.9783        flt_domestic
flt_intl_departures        flt_domestic 0.9777        flt_domestic
         soc_phuket    soc_phuket_beach 0.9615    soc_phuket_beach
   soc_phuket_beach    soc_phuket_hotel 0.9510    soc_phuket_hotel
     cal_n_holidays        cal_n_events 0.9432                 NaN
         soc_phuket    soc_phuket_hotel 0.9326    soc_phuket_hotel
flt_intl_departures          soc_phuket 0.9013 flt_intl_departures
  flt_intl_arrivals          soc_phuket 0.8992   flt_intl_arrivals
       flt_domestic          soc_phuket 0.8737        flt_domestic
         soc_phuket   soc_phuket_flight 0.8703   soc_phuket_flight
flt_intl_departures    soc_phuket_beach

In [10]:
# ── Export ml_dataset.csv (all 96 rows + lags + split) ───────────────────────
import os

ml_dataset = df.copy()

ml_dataset.to_csv('../final_data/tables/ml_dataset.csv', index=False)

print('=' * 70)
print('EXPORT: ml_dataset.csv')
print('=' * 70)
print(f'Shape: {ml_dataset.shape[0]} rows × {ml_dataset.shape[1]} cols')
print(f'Size: {os.path.getsize("../final_data/tables/ml_dataset.csv")/1024:.1f} KB')
print(f'New columns vs master: {[c for c in ml_dataset.columns if c not in pd.read_csv("../data/phuket_master.csv").columns]}')

print()
print('Split distribution in ml_dataset:')
print(ml_dataset['split'].value_counts().to_string())

print()
print('Target stats per split (tt_ratio_Weekday_AM1, pti_mean):')
for s in ['train', 'val', 'test']:
    sub = ml_dataset[ml_dataset['split'] == s]
    tt = sub['tt_ratio_Weekday_AM1']
    pti = sub['pti_mean']
    print(f'  {s:5s}: tt_ratio_am μ={tt.mean():.3f} σ={tt.std():.3f}  |  pti_mean μ={pti.mean():.3f} σ={pti.std():.3f}')

print()
print('✓ Saved: final_data/tables/ml_dataset.csv')

EXPORT: ml_dataset.csv
Shape: 96 rows × 132 cols
Size: 66.5 KB
New columns vs master: ['flt_total_pax_l1', 'flt_total_pax_l2', 'flt_total_pax_l3', 'wx_rain_mm_sum_l1', 'wx_rain_mm_sum_l2', 'wx_rain_mm_sum_l3', 'soc_phuket_l1', 'soc_phuket_l2', 'soc_phuket_l3', 'cal_n_holidays_l1', 'cal_n_holidays_l2', 'cal_n_holidays_l3', 'tt_ratio_Weekday_AM1_l1', 'tt_ratio_Weekday_AM1_l2', 'tt_ratio_Weekday_AM1_l3', 'pti_Weekday_AM1_l1', 'pti_Weekday_AM1_l2', 'pti_Weekday_AM1_l3', 'spd_Weekday_AM1_l1', 'spd_Weekday_AM1_l2', 'spd_Weekday_AM1_l3', 'tt_ratio_Weekday_AM2_l1', 'tt_ratio_Weekday_AM2_l2', 'tt_ratio_Weekday_AM2_l3', 'tt_ratio_Weekday_PrePM_l1', 'tt_ratio_Weekday_PrePM_l2', 'tt_ratio_Weekday_PrePM_l3', 'tt_ratio_Weekend_AM1_l1', 'tt_ratio_Weekend_AM1_l2', 'tt_ratio_Weekend_AM1_l3', 'split']

Split distribution in ml_dataset:
split
train    48
val      24
test     24

Target stats per split (tt_ratio_Weekday_AM1, pti_mean):
  train: tt_ratio_am μ=1.369 σ=0.228  |  pti_mean μ=2.699 σ=0.436
  va

In [11]:
# ── Create and export ML train / val / test CSVs ──────────────────────────────
# Drop rows with NaN lags (first 3 months of 2023 per corridor)
ml_clean = ml_dataset.dropna(subset=lag_cols).copy()

# Find true first row after dropping NaN lags
first_row = ml_clean.sort_values(['corr_id', 'year', 'month']).iloc[0]

print('=' * 70)
print('CREATING ML TRAIN / VAL / TEST SPLITS')
print('=' * 70)
print(f'After dropping NaN lag rows: {len(ml_clean)} (from {len(ml_dataset)})')
print(f'Dropped: {len(ml_dataset) - len(ml_clean)} rows (Jan/Feb/Mar 2023 × 4 corridors)')
print(f'First usable row: {int(first_row["year"])}-{int(first_row["month"]):02d}  (corr_id={int(first_row["corr_id"])})')

# Build export column list with CORRECT deduplication (preserves order)
raw_export_cols = ML_FEATURES + TARGETS + ['year', 'month', 'corr_id', 'corridor', 'season', 'split']
_seen2 = set()
EXPORT_COLS_ML = []
for c in raw_export_cols:
    if c not in _seen2:
        EXPORT_COLS_ML.append(c)
        _seen2.add(c)

print(f'Export columns: {len(EXPORT_COLS_ML)} unique (deduped from {len(raw_export_cols)} — targets also in traffic_features)')

for s in ['train', 'val', 'test']:
    sub = ml_clean[ml_clean['split'] == s][EXPORT_COLS_ML].copy()
    fname = f'../final_data/tables/ml_{s}.csv'
    sub.to_csv(fname, index=False)
    size_kb = os.path.getsize(fname) / 1024

    # Proper date range: use row-sorted first/last, not independent year.min/month.min
    sub_sorted = sub.sort_values(['year', 'month'])
    date_min = f"{int(sub_sorted.iloc[0]['year'])}-{int(sub_sorted.iloc[0]['month']):02d}"
    date_max = f"{int(sub_sorted.iloc[-1]['year'])}-{int(sub_sorted.iloc[-1]['month']):02d}"

    tt  = sub['tt_ratio_Weekday_AM1']
    pti = sub['pti_mean']
    print(f'\n--- {s.upper()} ---')
    print(f'  Shape    : {sub.shape[0]} rows × {sub.shape[1]} cols | {size_kb:.1f} KB')
    print(f'  Range    : {date_min} → {date_max}')
    print(f'  Corridors: {int(sub["corr_id"].nunique())}')
    print(f'  tt_ratio_Weekday_AM1 : mean={float(tt.mean()):.4f}  std={float(tt.std()):.4f}  min={float(tt.min()):.4f}  max={float(tt.max()):.4f}')
    print(f'  pti_mean         : mean={float(pti.mean()):.4f}  std={float(pti.std()):.4f}')
    print(f'  Seasons  : {dict(sub["season"].value_counts())}')
    print(f'  ✓ Saved: final_data/tables/ml_{s}.csv')

print()
print(f'Total ML rows: {len(ml_clean)}')
print(f'  Train: {int((ml_clean["split"]=="train").sum())} | Val: {int((ml_clean["split"]=="val").sum())} | Test: {int((ml_clean["split"]=="test").sum())}')

CREATING ML TRAIN / VAL / TEST SPLITS
After dropping NaN lag rows: 84 (from 96)
Dropped: 12 rows (Jan/Feb/Mar 2023 × 4 corridors)
First usable row: 2023-04  (corr_id=0)
Export columns: 124 unique (deduped from 131 — targets also in traffic_features)

--- TRAIN ---
  Shape    : 36 rows × 124 cols | 25.5 KB
  Range    : 2023-04 → 2023-12
  Corridors: 4
  tt_ratio_Weekday_AM1 : mean=1.3906  std=0.2381  min=0.9900  max=1.7900
  pti_mean         : mean=2.7244  std=0.4546
  Seasons  : {'monsoon': np.int64(20), 'shoulder': np.int64(12), 'high_peak': np.int64(4)}
  ✓ Saved: final_data/tables/ml_train.csv

--- VAL ---
  Shape    : 24 rows × 124 cols | 17.7 KB
  Range    : 2024-01 → 2024-06
  Corridors: 4
  tt_ratio_Weekday_AM1 : mean=1.4083  std=0.2214  min=1.0400  max=1.8400
  pti_mean         : mean=2.8632  std=0.5475
  Seasons  : {'high_peak': np.int64(12), 'shoulder': np.int64(8), 'monsoon': np.int64(4)}
  ✓ Saved: final_data/tables/ml_val.csv

--- TEST ---
  Shape    : 24 rows × 124 cols |

In [12]:
# ── Final validation summary ───────────────────────────────────────────────────
print('=' * 70)
print('PHASE 2.1 — FINAL VALIDATION SUMMARY')
print('=' * 70)

files_to_check = {
    'ml_dataset.csv' : '../final_data/tables/ml_dataset.csv',
    'ml_train.csv'   : '../final_data/tables/ml_train.csv',
    'ml_val.csv'     : '../final_data/tables/ml_val.csv',
    'ml_test.csv'    : '../final_data/tables/ml_test.csv',
}

all_ok = True
for name, path in files_to_check.items():
    if os.path.exists(path):
        tmp = pd.read_csv(path)
        nans = tmp.isna().sum().sum()
        size_kb = os.path.getsize(path) / 1024
        status = '✓' if nans == 0 else f'⚠ {nans} NaN'
        print(f'{status} {name}: {tmp.shape[0]}r × {tmp.shape[1]}c | {size_kb:.1f} KB | NaN={nans}')
    else:
        print(f'✗ MISSING: {name}')
        all_ok = False

ml_ds = pd.read_csv('../final_data/tables/ml_dataset.csv')
print(f'\nml_dataset NaN detail across lag columns:')
print(ml_ds[lag_cols].isna().sum().to_string())
print(f'Rows with any lag NaN: {ml_ds[lag_cols].isna().any(axis=1).sum()} / {len(ml_ds)}')

print()
print('ML split shapes summary:')
for s in ['train', 'val', 'test']:
    tmp = pd.read_csv(f'../final_data/tables/ml_{s}.csv')
    assert tmp.isna().sum().sum() == 0, f'{s}: has NaN!'
    print(f'  ml_{s}.csv: {tmp.shape} | NaN=0 ✓')

print()
print('PIPELINE STATUS ok:')
print('  ml_dataset.csv  → input for notebook 04_textualize (all 96 rows + exo/traffic lags)')
print('  ml_train.csv    → ML training / optional teacher baselines')
print('  ml_val.csv      → model selection / early stopping (Jan – Jun 2024)')
print('  ml_test.csv     → final evaluation (Jul – Dec 2024)')
print()
print('NOTE: Traffic features are DYNAMIC (real TomTom monthly values, 2023-2024).')
print('  → V9 adds forecast-critical traffic-memory lags on top of the original exogenous lags.')


PHASE 2.1 — FINAL VALIDATION SUMMARY
⚠ 240 NaN ml_dataset.csv: 96r × 132c | 66.5 KB | NaN=240
✓ ml_train.csv: 36r × 124c | 25.5 KB | NaN=0
✓ ml_val.csv: 24r × 124c | 17.7 KB | NaN=0
✓ ml_test.csv: 24r × 124c | 17.7 KB | NaN=0

ml_dataset NaN detail across lag columns:
flt_total_pax_l1              4
flt_total_pax_l2              8
flt_total_pax_l3             12
wx_rain_mm_sum_l1             4
wx_rain_mm_sum_l2             8
wx_rain_mm_sum_l3            12
soc_phuket_l1                 4
soc_phuket_l2                 8
soc_phuket_l3                12
cal_n_holidays_l1             4
cal_n_holidays_l2             8
cal_n_holidays_l3            12
tt_ratio_Weekday_AM1_l1       4
tt_ratio_Weekday_AM1_l2       8
tt_ratio_Weekday_AM1_l3      12
pti_Weekday_AM1_l1            4
pti_Weekday_AM1_l2            8
pti_Weekday_AM1_l3           12
spd_Weekday_AM1_l1            4
spd_Weekday_AM1_l2            8
spd_Weekday_AM1_l3           12
tt_ratio_Weekday_AM2_l1       4
tt_ratio_Weekday_AM2_l2    

## Summary

| File | Rows | Cols | Purpose |
|------|------|------|---------|
| `ml_dataset.csv` | 96 | varies | Full enriched table (master + lags + split) — input for textualization |
| `ml_train.csv` | ~36 | varies | XGBoost training — 2023 (NaN lags dropped) |
| `ml_val.csv` | 24 | varies | Validation — Jan–Jun 2024 |
| `ml_test.csv` | 24 | varies | Test — Jul–Dec 2024 |

**Feature reduction logic** (ML only):
- Dropped 8 multicollinear features vs LLM set
- Traffic features are dynamic (24 timesets × 24 months, real TomTom data)
- Lag L3 forces exclusion of Jan/Feb/Mar 2023 from ML training

**Next:** `04_textualize.ipynb` — generate 576 LLM prompts from `ml_dataset.csv`